# The error taxonomy — each failure names its cause

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/f-inverse/jammi-ai/blob/py-v0.49.1/cookbook/notebooks/book/17-channels-taxonomy/channels-taxonomy.ipynb)

Built from [`cookbook/book/chapters/17-channels-taxonomy/channels-taxonomy.qmd`](https://github.com/f-inverse/jammi-ai/blob/main/cookbook/book/chapters/17-channels-taxonomy/channels-taxonomy.qmd). Run the setup
cell first; every other cell runs top to bottom.

In [ ]:
# Setup: jammi 0.49.1 — the CUDA engine on an sm_80+ GPU (L4, A100, …), the
# CPU engine otherwise — and the cookbook's library and fixtures. The chapter runs
# at `small` scale, over the committed fixtures, in minutes. SCALE = "full" runs
# it over the published data and real encoders instead: meant for a GPU, and the
# chapters that fine-tune take hours there.
import os
import subprocess
import sys


def compute_capability() -> float:
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            capture_output=True, text=True, check=True,
        ).stdout.split()
    except (OSError, subprocess.CalledProcessError):
        return 0.0
    return float(out[0]) if out else 0.0


gpu = compute_capability() >= 8.0
engine = "jammi-ai-native-cu12" if gpu else "jammi-ai-native"
server = "jammi-server-cu12" if gpu else "jammi-server"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jammi-ai==0.49.1", engine + "==0.49.1", "jammi-cookbook==0.49.1", server + "==0.49.1"], check=True)
SCALE = "small"
os.environ["JAMMI_COOKBOOK_SCALE"] = SCALE
print(f"engine: {engine}   scale: {SCALE}")

In [ ]:
import jammi_cookbook

**Recipe:** `register_channel` · `add_channel_columns` · `list_channels` ·
`jammi.errors` · **Theory:** a typed error vocabulary — a failure with a known
cause names it, as a gRPC status code on the wire (The gRPC Authors 2024) and as a
Python class in the client, so a caller branches on the cause instead of parsing
a message · **Rail:** parity (each failure raises the same class in process and
from a server, and the server's status code names the same condition).

Chapter 14 measured the evidence-channel registry's happy path. This chapter
measures its failures — and, through them, the error vocabulary every engine
verb shares. A failure with a known cause is never a generic fault. It raises a
`jammi.errors` class that names the condition, and over the wire it carries the
gRPC status code for that condition:

| condition | class | status code |
|---|---|---|
| the argument is malformed | `InvalidArgument` | `INVALID_ARGUMENT` |
| the thing named does not exist | `NotFound` | `NOT_FOUND` |
| the thing being created exists already | `AlreadyExists` | `ALREADY_EXISTS` |
| the state the operation needs does not hold | `FailedPrecondition` | `FAILED_PRECONDITION` |

A more specific refusal refines one of these — `ModelNotFound` is a `NotFound`,
`ModelReferenced` a `FailedPrecondition` — so a caller catches as narrowly as it
needs to. Everything else is a `BackendError`, the class all four refine: the
residual for a fault with no named cause.

In [ ]:
import tempfile
import uuid

import jammi
from jammi.errors import AlreadyExists, FailedPrecondition, InvalidArgument, NotFound
from jammi.testing import LiveServer

CONDITIONS = (InvalidArgument, NotFound, AlreadyExists, FailedPrecondition)

## Four failures, written once

Each failure is a real misuse of the registry: registering a channel id twice,
extending a channel never registered, redeclaring a column with another type,
and registering an empty channel id.

In [ ]:
def failure(action) -> tuple[str, str | None]:
    """The class an action raises, and the status code when it came over the wire."""
    try:
        action()
    except CONDITIONS as raised:
        code = getattr(raised, "code", None)
        return type(raised).__name__, code.name if code is not None else None
    raise AssertionError("the action was expected to fail")


def failures(db) -> dict:
    with db.tenant_scope(str(uuid.uuid4())):
        db.register_channel("scored_by", priority=50, columns=[("score", "Float64")])
        return {
            "register twice": failure(lambda: db.register_channel(
                "scored_by", priority=50, columns=[("score", "Float64")])),
            "extend an unregistered channel": failure(lambda: db.add_channel_columns(
                "never_registered", columns=[("note", "Utf8")])),
            "redeclare a column's type": failure(lambda: db.add_channel_columns(
                "scored_by", columns=[("score", "Utf8")])),
            "an empty channel id": failure(lambda: db.register_channel(
                "", priority=1, columns=[("x", "Utf8")])),
        }

## In process, then against a server

In [ ]:
with jammi.connect(f"file://{tempfile.mkdtemp()}") as db:
    local = failures(db)
with LiveServer(tempfile.mkdtemp()) as server, jammi.connect(server.endpoint) as db:
    served = failures(db)

print(f"{'failure':<32}{'class (in process)':<22}{'class (server)':<22}{'status code'}")
for name in local:
    print(f"{name:<32}{local[name][0]:<22}{served[name][0]:<22}{served[name][1]}")

In [ ]:
expected = {
    "register twice": ("AlreadyExists", "ALREADY_EXISTS"),
    "extend an unregistered channel": ("NotFound", "NOT_FOUND"),
    "redeclare a column's type": ("FailedPrecondition", "FAILED_PRECONDITION"),
    "an empty channel id": ("InvalidArgument", "INVALID_ARGUMENT"),
}
for name, (cls, code) in expected.items():
    assert local[name][0] == cls and served[name] == (cls, code), name

Every failure raises the same class on both transports, and the server's status
code names the same condition. A program that catches `AlreadyExists` to treat
a repeated registration as done, or `NotFound` to register a channel on first
use, works unchanged in process and against a server — and neither ever reads a
message.

## Bridge note

> **A cause is a type, not a sentence.** The engine names each failure it can
> explain — malformed, missing, already there, not in the required state — with
> a gRPC status code on the wire (The gRPC Authors 2024) and the matching
> `jammi.errors` class in the client, refined by resource where a caller needs
> more (`ModelReferenced`). The classes are the same on both transports because
> both derive them from the one classification the server sends.

## References

- The gRPC Authors (2024) *Status Codes* gRPC documentation, https://grpc.io/docs/guides/status-codes/.